Import libraries and load dataset

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

df = pd.read_csv("../datasets/ecommerce_return_prediction_feature_engineered.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df["returned"].value_counts())

Prepare the data

In [ ]:
df = df.drop_duplicates()

X = df.drop(columns=["returned", "return_rate_pct"])
y = df["returned"]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))
print("Features prepared successfully")

Split data and create preprocessing pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Define four models and hyperparameter grids

In [ ]:
models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {
            "model__C": [0.1, 1, 10],
            "model__solver": ["liblinear", "lbfgs"]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {
            "model__max_depth": [5, 10, 20, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 5]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [10, 20, None],
            "model__min_samples_leaf": [1, 2, 5]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5],
            "model__learning_rate": [0.05, 0.1]
        }
    )
}

print("Four models configured successfully")

Train models and tune hyperparameters

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = {}
tuning_results = []

for name, (model, params) in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    search = GridSearchCV(
        pipeline,
        param_grid=params,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_

    tuning_results.append({
        "Model": name,
        "Best Parameters": search.best_params_,
        "Best Cross-Validation F1": search.best_score_
    })

    print("\nModel:", name)
    print("Best parameters:", search.best_params_)
    print("Best CV F1:", round(search.best_score_, 4))

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df)